# 1D Multiscale RM-CLEAN

*NOTE:* This is still under development and experimentation. Consider this mode experimental.

Standard RM-CLEAN models the Faraday dispersion function (FDF) as a sum of delta functions convolved with the RMSF. That is ideal for Faraday-thin (point) sources, but a Faraday-thick source is genuinely extended in Faraday depth, and a pile of deltas is a poor model of it: the components form a spiky comb that overshoots the true envelope, even though the restored FDF looks fine once the beam smooths it over.

Multiscale RM-CLEAN is the RM-domain analogue of image-plane multiscale CLEAN (Cornwell 2008; Offringa & Smirnov 2017). Each component is a tapered kernel at one of several Faraday-depth *scales* (in units of the RMSF FWHM), not just a delta. What it buys you is a better *component model* of thick structure, and on thick sources it also tends to need fewer components: single-scale must build a deep comb of deltas to represent the width, while multiscale traces it with a handful of wide kernels. Every sub-minor iteration still places one component at a fixed gain, so the honest cost counter is `n_sub_minor_iter`.

Scale selection is width-gated: a point-like peak is cleaned exactly like single-scale CLEAN (point safety), while a genuinely wide peak engages the extended scales via matched-filter scoring. Turn the whole thing on with `multiscale=True`.

## How a scale gets chosen

At every step CLEAN has to decide which scale to place the next component on. The obvious score for "how well does scale $s$ explain the current residual" is the peak of the residual $R$ convolved with that scale's kernel $K_s$,

$$\mathrm{peak}_s = \max_\phi \, \lvert R * K_s \rvert .$$

The problem is that a wider kernel sums over more of the FDF, and the noise in an FDF is strongly correlated (the RMSF has broad wings), so a wide kernel piles up correlated noise and its raw peak is inflated even on pure noise. Comparing raw peaks would always favour the widest scale.

### Why not the Offringa & Smirnov (2017) bias

O&S17 solve this for image-plane multiscale CLEAN with a fixed *scale bias*: multiply each score by a factor that geometrically down-weights larger scales,

$$\mathrm{score}_s = \mathrm{peak}_s \cdot \beta^{-1-\log_2(s/s_0)},$$

with one tuning constant $\beta$ (WSClean uses 0.6). That is fine when the noise is white and roughly the same on every scale, as it is in an image. It does not describe the RM domain: here the per-scale noise penalty is set by the RMSF shape, not by a clean geometric law, so no single $\beta$ can match it. On the deep scale grids used here a fixed bias mis-weights the scales and pulls point sources onto oversized kernels, which blows up their recovered flux. We tried an exact re-implementation; it is not point-safe here, which is why rm-lite does not use it.

### Matched-filter scoring instead

Rather than guess a penalty, we normalise each score by the noise the kernel actually lets through. A few definitions make this precise.

**Correlated FDF noise.** Independent noise in the measured $Q$ and $U$ channels does not stay independent after RM synthesis: every point of the FDF is a weighted sum of the same channels, so neighbouring Faraday depths share noise. The shape of that correlation is exactly the RMSF, so the RMSF plays the role of the noise *covariance* of the FDF. This is the root of the problem above: a wide kernel averages over many correlated cells, and correlated noise does not average down the way independent noise would.

**Scale kernel $K_s$.** The tapered kernel for scale $s$ (in RMSF FWHM units), sum-normalised so that scale 0 is a delta and cleaning on it is ordinary single-scale CLEAN. The smoothed residual $R * K_s$ is what the score looks at, and $\max_\phi \lvert R * K_s\rvert$ is the numerator above.

**Per-scale noise level $\sigma_s$.** Because the covariance is the RMSF, the standard deviation of $\lvert R * K_s\rvert$ on a noise-only spectrum has a closed form,

$$\sigma_s = \sqrt{\frac{(K_s * K_s * \mathrm{RMSF})(0)}{\mathrm{RMSF}(0)}}, \qquad \sigma_0 = 1 .$$

Reading it left to right: $*$ is convolution; $K_s * K_s$ applies the kernel twice (once for the kernel in the score, once because the noise itself is pushed through the same filter); convolving that with the RMSF propagates the noise covariance through the filter; and $(\,0\,)$ evaluates the result at zero lag (the RMSF peak channel), which is the variance the filter passes. Dividing by $\mathrm{RMSF}(0)$ measures everything in units of the ordinary single-scale noise, which is what fixes $\sigma_0 = 1$. So $\sigma_s$ is simply "how many times noisier than a single-scale clean is scale $s$".

**Matched filter.** A matched filter correlates the data with the template you are looking for and normalises by the noise that template picks up. Here the template is the scale kernel, so dividing the kernel response by its own noise level $\sigma_s$ is exactly that:

$$\mathrm{score}_s = \frac{\max_\phi \lvert R * K_s \rvert}{\sigma_s}.$$

The numerator measures how strongly scale $s$ responds to real structure in the residual; the denominator measures how strongly it responds to noise alone. Their ratio is a detection signal-to-noise, directly comparable across scales with no constant to hand-tune. A scale wins only when it detects structure above its *own* noise floor, not merely because it is wide.

### Width gating keeps points safe

A raw signal-to-noise argmax is still slightly scale-degenerate under correlated noise, so on its own it rarely engages a wide scale in a useful way. We gate it on the width of the feature being cleaned. An extended scale is engaged only when both:

1. the residual peak is genuinely wider than the beam: its half-max width exceeds $1.2\times$ the measured RMSF (dirty-beam) width, where a point source sits at $1.0\times$; and
2. the best extended score is at least $0.85\times$ the delta-scale score, so structure that happens to be wide but is poorly matched to any wide kernel does not trigger it.

If either test fails the step falls back to the signal-to-noise score, and among scales within a small margin (`multiscale_selection_margin`, default 0.08) of the best it takes the *smallest*. The result is point safety: a thin source is cleaned exactly like single-scale CLEAN, and wide scales activate only on structure that is really resolved.

## How masking works

Masking happens in stages, and none of them is a global cut on the raw FDF.

First a **single-scale pass** cleans the spectrum on the delta scale alone. Wherever it places components defines the *source region*: a tight footprint around real emission. A noise-only spectrum leaves this empty, so multiscale CLEAN then does nothing, which is exactly what we want.

Each scale then gets **its own mask** through an opening filter. The dirty FDF is convolved with the scale kernel, morphologically opened (erode then dilate by the scale width) so only structure at least that wide survives, thresholded at the flux-correct level $\mathrm{mask}\times$(kernel peak response), and finally confined to the source region dilated by the scale width. That dilation is what stops a wide kernel leaking onto the RMSF sidelobe islands that sit off to the side of a bright source. A scale whose mask comes out empty is switched off for this spectrum.

Cleaning is then two-phase. Phase 1 cleans down to the mask level and records where each scale actually placed flux. Phase 2 deep-cleans to the final threshold, but restricts each scale to its own phase-1 footprint, so nothing smears fresh flux across the window late in the clean.

In [ ]:
from __future__ import annotations

import logging

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import quantity_support
from rm_lite.tools_1d import rmclean, rmsynth
from rm_lite.utils.fitting import unit_centred_gaussian
from rm_lite.utils.logging import logger
from rm_lite.utils.simulate import build_model_fdf, delta, gauss, simulate_fdf
from rm_lite.utils.synthesis import (
    calc_faraday_moments,
    freq_to_lambda2,
)

plt.rcParams["figure.dpi"] = 150
_ = quantity_support()
logger.setLevel(logging.WARNING)
rng = np.random.default_rng(42)

# A wide fractional bandwidth (300-1800 MHz) so extended Faraday structure
# survives depolarisation over several RMSF widths.
freqs = np.linspace(300, 1800, 300) * u.MHz
freq_hz = freqs.to(u.Hz).value
sigma_chan = 0.02  # per-channel noise in Q and U


def run_synth(complex_pol_arr):
    return rmsynth.run_rmsynth(
        freq_arr_hz=freq_hz,
        complex_pol_arr=complex_pol_arr,
        complex_pol_error=np.ones_like(complex_pol_arr)
        * (sigma_chan + 1j * sigma_chan),
        n_samples=50,
        phi_max_radm2=100,
    )

## Example 1: Faraday-thin source (point safety)

A thin source at RM = +3.3 RMSF FWHM. Multiscale must reproduce single-scale CLEAN here: same flux, same number of component-placement steps, all components on the delta scale at the source RM. The `simulate_fdf` helper forward-models the source through channel space, so the noise in the FDF is the physically correct correlated field.

In [ ]:
thin_spec = delta(center_fwhm=3.3, amp=1.0)
sim_thin = simulate_fdf(thin_spec, freq_hz, rng=rng, sigma=sigma_chan)
synth_thin = run_synth(sim_thin.complex_pol_arr)

fwhm = float(synth_thin.fdf_parameters["fwhm_rmsf_radm2"][0])
rm_true = 3.3 * fwhm * u.rad / u.m**2
print(f"RMSF FWHM = {fwhm:.2f} rad/m2, source RM = {rm_true:.1f}")

In [ ]:
help(rmclean.run_rmclean_from_synth)

In [ ]:
single_thin = rmclean.run_rmclean_from_synth(synth_thin, auto_mask=7, auto_threshold=1)
multi_thin = rmclean.run_rmclean_from_synth(
    synth_thin, auto_mask=7, auto_threshold=1, multiscale=True
)


def unpack(res):
    arrs = res.fdf_arrs
    params = res.clean_parameters
    return {
        "phi": arrs["phi_arr_radm2"].to_numpy() * u.rad / u.m**2,
        "dirty": arrs["fdf_dirty_complex_arr"].to_numpy().astype(complex),
        "clean": arrs["fdf_clean_complex_arr"].to_numpy().astype(complex),
        "model": arrs["fdf_model_complex_arr"].to_numpy().astype(complex),
        "resid": arrs["fdf_residual_complex_arr"].to_numpy().astype(complex),
        "mask": float(params["mask"][0]),
        "threshold": float(params["threshold"][0]),
        "n_iter": int(params["n_iter"][0]),
        "n_sub_minor_iter": int(params["n_sub_minor_iter"][0]),
    }


s = unpack(single_thin)
m = unpack(multi_thin)

In [ ]:
fig, ax = plt.subplots()
ax.plot(m["phi"], np.abs(m["dirty"]), color="k", alpha=0.4, label="dirty")
ax.plot(
    m["phi"],
    np.abs(m["clean"] - m["resid"]),
    color="k",
    alpha=1,
    label="convolved model",
)
ax.step(m["phi"], np.abs(m["model"]), color="tab:red", where="mid", label="model")
ax.plot(
    m["phi"], np.abs(m["resid"]), color="tab:blue", alpha=1, zorder=10, label="residual"
)
ax.axhline(m["threshold"], color="k", ls=":", label="threshold", lw=1)
ax.axhline(m["mask"], color="k", ls="--", label="mask", lw=1)
ax.set(
    xlabel=rf"$\phi$ / {u.rad / u.m**2:latex_inline}",
    ylabel="polarised intensity",
    title="Thin source, multiscale CLEAN",
    xlim=(rm_true.value - 15 * fwhm, rm_true.value + 15 * fwhm),
    yscale="log",
    ylim=(1e-5, 2e0),
)
ax.legend()

The component model is a handful of deltas at the source RM, exactly as single-scale would place them. The assertions below check the guarantees rather than describing them: the model flux matches single-scale, the components sit at the source RM, and the component-placement count is identical.

One counter subtlety: `n_iter` means different things in the two modes. For single-scale it counts components; for multiscale it counts *minor cycles* (scale re-selections), each of which can place many components. `n_sub_minor_iter` is the component count in both modes, so that is the number to compare.

In [ ]:
# Components sit at the source RM.
support = np.abs(m["model"]) > 0
assert support.any()
assert np.all(np.abs(m["phi"][support] - rm_true) < 1.5 * fwhm * u.rad / u.m**2)

# Flux parity with single-scale.
m0_single = float(calc_faraday_moments(np.abs(s["clean"]), s["phi"].value, fwhm).mom0)
m0_multi = float(calc_faraday_moments(np.abs(m["clean"]), m["phi"].value, fwhm).mom0)
assert np.isclose(m0_single, m0_multi, rtol=1e-3)

# Identical work: same number of component placements.
assert m["n_sub_minor_iter"] == s["n_sub_minor_iter"]

print(f"flux: single {m0_single:.3f}, multi {m0_multi:.3f}")
print(
    f"components placed: single {s['n_sub_minor_iter']}, multi {m['n_sub_minor_iter']}"
)
print(f"multiscale minor cycles (scale re-selections): {m['n_iter']}")

## Example 2: Faraday-thick source

An external-dispersion Gaussian with sigma = 1.5 RMSF FWHM. On this band such a source is still polarised over much of the band, so its width is genuinely recoverable.

This is where the two algorithms differ. Both recover the flux and both leave a comparable residual, but the *component models* are very different: single-scale builds a comb of deltas that overshoots the true envelope, while multiscale builds the source from wide kernels that trace it. The four panels show the input model and the clean models, bare and convolved with the clean beam. The beam hides the difference, which is exactly why the model matters: any physical interpretation of Faraday thickness uses the model, not the restored FDF.

Because single-scale has to represent the width with a deep comb of deltas, it also places far *more* components than multiscale, which traces the source with a few wide kernels. So on a thick source multiscale wins on both counts, model shape and component count, checked below.

In [ ]:
thick_spec = gauss(1.5, amp=1.0)
sim_thick = simulate_fdf(thick_spec, freq_hz, rng=rng, sigma=sigma_chan)
synth_thick = run_synth(sim_thick.complex_pol_arr)

single_thick = rmclean.run_rmclean_from_synth(
    synth_thick, auto_mask=10, auto_threshold=1
)
multi_thick = rmclean.run_rmclean_from_synth(
    synth_thick,
    auto_mask=10,
    auto_threshold=1,
    multiscale=True,
)
s = unpack(single_thick)
m = unpack(multi_thick)

print(
    f"components placed: single {s['n_sub_minor_iter']}, multi {m['n_sub_minor_iter']}"
)
print(f"multiscale minor cycles (scale re-selections): {m['n_iter']}")

phi = m["phi"].value
truth_model = build_model_fdf(thick_spec, phi, fwhm)

# Unit-peak clean beam, matching how rm_lite restores the model.
beam = unit_centred_gaussian(phi - phi[phi.size // 2], fwhm=fwhm)


def conv_beam(arr):
    return np.convolve(arr.real, beam, mode="same") + 1j * np.convolve(
        arr.imag, beam, mode="same"
    )

In [ ]:
fig, axs = plt.subplots(2, 2, sharex="col", sharey="row", figsize=(12, 6))


axs[0, 0].plot(
    phi, np.abs(s["clean"] - s["resid"]), color="k", alpha=1, label="convolved model"
)
axs[0, 0].plot(
    phi, np.abs(s["resid"]), color="tab:blue", alpha=1, zorder=10, label="residual"
)
axs[0, 0].step(
    phi, np.abs(s["model"]), color="tab:red", where="mid", label="model", lw=0.5
)

axs[0, 0].axhline(m["threshold"], color="k", ls=":", label="threshold", lw=1)
axs[0, 0].axhline(m["mask"], color="k", ls="--", label="mask", lw=1)
axs[0, 0].legend()
axs[0, 0].set_title("Delta scale")
axs[0, 0].set(ylabel="polarised intensity")


axs[0, 1].plot(phi, np.abs(m["clean"] - m["resid"]), color="k", alpha=1)
axs[0, 1].plot(phi, np.abs(m["resid"]), color="tab:blue", alpha=1, zorder=10)
axs[0, 1].step(
    phi, np.abs(m["model"]), color="tab:red", where="mid", label="Single-scale", lw=0.5
)
axs[0, 1].axhline(m["threshold"], color="k", ls=":", label="Threshold", lw=1)
axs[0, 1].axhline(m["mask"], color="k", ls="--", label="Mask", lw=1)
axs[0, 1].set_title("Multi-scale")

axs[1, 0].plot(phi, np.abs(s["resid"]), color="tab:blue", zorder=10)
axs[1, 0].step(
    phi, np.abs(s["model"]), color="tab:red", where="mid", label="Single-scale", lw=0.5
)
axs[1, 0].axhline(m["threshold"], color="k", ls=":", label="Threshold", lw=1)
axs[1, 0].axhline(m["mask"], color="k", ls="--", label="Mask", lw=1)
axs[1, 0].set(
    ylabel="polarised intensity", xlabel=rf"$\phi$ / {u.rad / u.m**2:latex_inline}"
)

axs[1, 1].plot(phi, np.abs(m["resid"]), color="tab:blue", zorder=10)
axs[1, 1].step(
    phi, np.abs(m["model"]), color="tab:red", where="mid", label="Single-scale", lw=0.5
)
axs[1, 1].axhline(m["threshold"], color="k", ls=":", label="Threshold", lw=1)
axs[1, 1].axhline(m["mask"], color="k", ls="--", label="Mask", lw=1)
axs[1, 1].set(
    yscale="log", ylim=(1e-4, 2e-1), xlabel=rf"$\phi$ / {u.rad / u.m**2:latex_inline}"
)

fig.suptitle("Faraday thick source")

In [ ]:
def model_shape_error(model, truth):
    """Scale-free rms of |model| against best-fit-amplitude |truth|."""
    a, t = np.abs(model), np.abs(truth)
    amp = float((a * t).sum() / (t * t).sum())
    return float(np.sqrt(np.mean((a - amp * t) ** 2)) / (amp * t.max()))


err_single = model_shape_error(s["model"], truth_model)
err_multi = model_shape_error(m["model"], truth_model)
print(
    f"model shape error: single {err_single:.3f}, multi {err_multi:.3f} "
    f"(ratio {err_multi / err_single:.2f})"
)

# The multiscale model is much closer to the true envelope.
assert err_multi < 0.5 * err_single

# At no cost in flux ...
m0_single = float(calc_faraday_moments(np.abs(s["clean"]), phi, fwhm).mom0)
m0_multi = float(calc_faraday_moments(np.abs(m["clean"]), phi, fwhm).mom0)
assert 0.9 < m0_multi / m0_single < 1.2

# ... and with fewer components: single-scale needs a deep comb of deltas to
# represent the width, while multiscale traces it with a handful of wide kernels.
steps_ratio = m["n_sub_minor_iter"] / s["n_sub_minor_iter"]
assert steps_ratio < 1.0
print(f"flux: single {m0_single:.3f}, multi {m0_multi:.3f}")
print(
    f"components placed: single {s['n_sub_minor_iter']}, multi {m['n_sub_minor_iter']} "
    f"(ratio {steps_ratio:.2f})"
)

## Example 3: narrow bands degenerate to single-scale

The scales are in units of the RMSF FWHM, and the largest recoverable Faraday scale is set by the shortest wavelength in the band: `phi_max_scale = pi / lambda_sq_min`. On a narrow band the RMSF is so wide that no extended scale fits below that ceiling, so the automatic grid collapses to `[0.0]` (with a warning) and multiscale CLEAN is exactly single-scale CLEAN. This is safe, and it is also physics: a source wider than the ceiling is depolarised out of the band, so there is nothing for a wide kernel to recover.

In [ ]:
from rm_lite.utils.clean import MultiscaleOptions, default_scales
from rm_lite.utils.synthesis import get_fwhm_rmsf, make_phi_arr

narrow_hz = np.linspace(744e6, 1032e6, 288)  # RACS-low band
lam2 = freq_to_lambda2(narrow_hz)
narrow_fwhm = float(get_fwhm_rmsf(lam2).fwhm_rmsf_radm2)
phi_max_scale = float(np.pi / lam2.min())
phi_narrow = make_phi_arr(500.0, narrow_fwhm / 10)

grid = default_scales(
    phi_narrow, narrow_fwhm, MultiscaleOptions(), phi_max_scale_radm2=phi_max_scale
)
print(f"RMSF FWHM = {narrow_fwhm:.1f} rad/m2")
print(
    f"largest recoverable scale = {phi_max_scale:.1f} rad/m2 "
    f"= {phi_max_scale / narrow_fwhm:.2f} FWHM"
)
print(f"auto scale grid (FWHM units): {grid}")

# No extended scale fits below the ceiling: the grid is delta-only.
assert grid.tolist() == [0.0]

In [ ]:
# And the clean itself is identical to single-scale, at identical cost.
sim_narrow = simulate_fdf(thin_spec, narrow_hz, rng=rng, sigma=sigma_chan)
synth_narrow = rmsynth.run_rmsynth(
    freq_arr_hz=narrow_hz,
    complex_pol_arr=sim_narrow.complex_pol_arr,
    complex_pol_error=np.ones_like(sim_narrow.complex_pol_arr)
    * (sigma_chan + 1j * sigma_chan),
    n_samples=50,
)
single_n = unpack(
    rmclean.run_rmclean_from_synth(synth_narrow, auto_mask=7, auto_threshold=1)
)
multi_n = unpack(
    rmclean.run_rmclean_from_synth(
        synth_narrow, auto_mask=7, auto_threshold=1, multiscale=True
    )
)
assert np.allclose(single_n["clean"], multi_n["clean"])
assert single_n["n_sub_minor_iter"] == multi_n["n_sub_minor_iter"]
print(
    "narrowband multiscale == single-scale, "
    f"{multi_n['n_sub_minor_iter']} components each"
)

### Setting scales manually

`multiscale_scales` overrides the automatic grid. The legitimate use is on a band that *does* support extended structure, when you want a leaner grid than the automatic one. Forcing extended scales onto a narrow band cannot beat the depolarisation ceiling: what little a wide kernel sees still fits at beam width, so the width-gated selection cleans it on the delta scale anyway.

One caution: a bigger scale is not a better scale. Under the correlated noise of the FDF the matched-filter score of a very wide kernel never competes with the delta scale, so forcing a grid like `[0, 8]` just disengages multiscale. The scale the automatic run actually engages on this source is 3, so the leaner grid worth forcing is `[0, 3]`. The model win survives with it.

In [ ]:
multi_forced = unpack(
    rmclean.run_rmclean_from_synth(
        synth_thick,
        auto_mask=10,
        auto_threshold=1,
        multiscale=True,
        multiscale_scales=np.array([0.0, 3.0]),
    )
)
err_forced = model_shape_error(multi_forced["model"], truth_model)
print(
    f"model shape error with forced grid [0, 3]: {err_forced:.3f} "
    f"(single-scale {err_single:.3f})"
)
assert err_forced < 0.7 * err_single

# Flux is still consistent with single-scale.
m0_forced = float(calc_faraday_moments(np.abs(multi_forced["clean"]), phi, fwhm).mom0)
assert 0.9 < m0_forced / m0_single < 1.2

In [ ]:
fig, ax = plt.subplots()
ax.plot(
    multi_forced["phi"],
    np.abs(multi_forced["dirty"]),
    color="k",
    alpha=0.4,
    label="dirty",
)
ax.plot(
    multi_forced["phi"],
    np.abs(multi_forced["clean"] - multi_forced["resid"]),
    color="k",
    alpha=1,
    label="convolved model",
)
ax.step(
    multi_forced["phi"],
    np.abs(multi_forced["model"]),
    color="tab:red",
    where="mid",
    label="model",
)
ax.plot(
    multi_forced["phi"],
    np.abs(multi_forced["resid"]),
    color="tab:blue",
    alpha=1,
    zorder=10,
    label="residual",
)
ax.axhline(multi_forced["threshold"], color="k", ls=":", label="threshold", lw=1)
ax.axhline(multi_forced["mask"], color="k", ls="--", label="mask", lw=1)
ax.set(
    xlabel=rf"$\phi$ / {u.rad / u.m**2:latex_inline}",
    ylabel="polarised intensity",
    title="Fixed scales [0, 3]",
    yscale="log",
    ylim=(1e-4, 2e0),
)
ax.legend()

## Summary

- `multiscale=True` is safe to leave on: thin sources are cleaned exactly like single-scale CLEAN, and narrow bands degenerate to single-scale by construction.
- The win is the *component model* of Faraday-thick structure; on thick sources multiscale also places far fewer components than single-scale, which needs a deep delta comb. Compare costs with `n_sub_minor_iter`; `n_iter` counts scale re-selections in multiscale mode.
- The largest recoverable Faraday scale is `pi / lambda_sq_min`. Wider structure is depolarised out of the band and no algorithm can recover it.
- `multiscale_scales` forces a custom grid (in RMSF FWHM units); it cannot beat the depolarisation ceiling, and very wide scales simply disengage.